# Shrub Lists Extras — QC Dashboard

Run **target/reference QC** on standardized shrub lists.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
STD_ORIG = Path('standardized') / 'shrubs_original_standardized.csv'
STD_REV  = Path('standardized') / 'shrubs_revised_standardized.csv'

shrubs_orig = pd.read_csv(STD_ORIG)
shrubs_rev  = pd.read_csv(STD_REV)
shrubs_all = pd.concat([shrubs_orig, shrubs_rev], ignore_index=True)
print('orig:', shrubs_orig.shape, 'rev:', shrubs_rev.shape, 'all:', shrubs_all.shape)


In [ ]:
def qc_missingness(df):
    return pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_frac': df.isna().mean(),
        'dtype': df.dtypes.astype(str),
    }).sort_values('missing_frac', ascending=False)

qc_missingness(shrubs_all)


In [ ]:
def qc_numeric_summary(df, cols=None):
    if cols is None:
        cols = ['x','y','z','height_m','radius_m','diameter_m','area_m2','volume_m3']
    cols = [c for c in cols if c in df.columns]
    return df[cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

qc_numeric_summary(shrubs_all)


In [ ]:
def qc_implausible_rows(df):
    checks = pd.DataFrame(index=df.index)
    if 'height_m' in df.columns:
        checks['height_nonpositive'] = df['height_m'].fillna(np.inf) <= 0
        checks['height_too_large'] = df['height_m'].fillna(-np.inf) > 15
    if 'diameter_m' in df.columns:
        checks['diam_nonpositive'] = df['diameter_m'].fillna(np.inf) <= 0
        checks['diam_too_large'] = df['diameter_m'].fillna(-np.inf) > 20
    if 'radius_m' in df.columns:
        checks['radius_nonpositive'] = df['radius_m'].fillna(np.inf) <= 0
    if {'x','y'}.issubset(df.columns):
        checks['missing_xy'] = df[['x','y']].isna().any(axis=1)
    checks['any_flag'] = checks.any(axis=1)
    return pd.concat([df, checks], axis=1)

flagged = qc_implausible_rows(shrubs_all)
flagged[flagged['any_flag']].head()


In [ ]:
def workflow_comparison(orig, rev):
    rows = []
    for name, df in [('original', orig), ('revised', rev)]:
        rows.append({
            'workflow': name,
            'n_shrubs': len(df),
            'n_with_xy': int(df[['x','y']].notna().all(axis=1).sum()) if {'x','y'}.issubset(df.columns) else np.nan,
            'height_mean': float(df['height_m'].mean()) if 'height_m' in df.columns else np.nan,
            'height_median': float(df['height_m'].median()) if 'height_m' in df.columns else np.nan,
            'diameter_mean': float(df['diameter_m'].mean()) if 'diameter_m' in df.columns else np.nan,
            'diameter_median': float(df['diameter_m'].median()) if 'diameter_m' in df.columns else np.nan,
        })
    return pd.DataFrame(rows).set_index('workflow')

workflow_comparison(shrubs_orig, shrubs_rev)


In [ ]:
plt.figure()
if 'height_m' in shrubs_orig.columns:
    plt.hist(shrubs_orig['height_m'].dropna(), bins=40, alpha=0.5, label='original')
if 'height_m' in shrubs_rev.columns:
    plt.hist(shrubs_rev['height_m'].dropna(), bins=40, alpha=0.5, label='revised')
plt.title('Shrub height distributions')
plt.xlabel('height_m')
plt.ylabel('count')
plt.legend()
plt.show()


In [ ]:
if {'x','y'}.issubset(shrubs_all.columns):
    plt.figure()
    plt.scatter(shrubs_orig['x'], shrubs_orig['y'], s=5, alpha=0.5, label='original')
    plt.scatter(shrubs_rev['x'], shrubs_rev['y'], s=5, alpha=0.5, label='revised')
    plt.title('Shrub centroid locations')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.show()
